In [53]:
from reader import EpubBilingualParser
parser = EpubBilingualParser("./data")
data = parser.get_all_books_data()

In [11]:
import spacy
from fuzzywuzzy import fuzz

# Загружаем модели для англ и рус
nlp_en = spacy.load("en_core_web_sm")
nlp_ru = spacy.load("ru_core_news_sm")

In [30]:
!pip uninstall comet

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/15.8 MB 4.8 MB/s eta 0:00:04
   ----- ---------------------------------- 2.1/15.8 MB 5.1 MB/s eta 0:00:03
   ------- -------------------------------- 3.1/15.8 MB 5.3 MB/s eta 0:00:03
   --------- ------------------------------ 3.9/15.8 MB 4.9 MB/s eta 0:00:03
   ------------ --------------------------- 5.0/15.8 MB 4.7 MB/s eta 0:00:03
   --------------- ------------------------ 6.0/15.8 MB 4.8 MB/s eta 0:00:03
   ----------------- ---------------------- 6.8/15.8 MB 4.8 MB/s eta 0:00:02
   ------------------- -------------------- 7.9/15.8 MB 4.8 MB/s eta 0:00:02
   ---------------------- ----------------- 8.9/15.8 MB 4.9 MB/s eta 0:00:02
   ------------------------- -------------- 10.2/15.8 MB 4.9 MB/s eta 0:00:02
   --------------------------- ------------ 10.7/15.8 MB 4.8 MB/s eta 0:00:02
   ----------------------------- ---------- 11.8/15.8 MB 4.8 MB/s eta 0:00:01
   

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.39.1 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
!pip install unbabel-comet


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import os
import re

def get_agent_translation(book_name: str, agent_index: int, chapter_name: str) -> str:
    """
    Находит и читает сохраненный файл перевода для конкретной главы и агента.
    """
    # 1. Очищаем имена точно так же, как при сохранении
    clean_book = re.sub(r'[\\/*?:"<>|]', "", book_name).strip()
    clean_chap = re.sub(r'[\\/*?:"<>|]', "", chapter_name).strip()
    
    # 2. Собираем путь: output / Название_Книги / Номер_Агента / Название_Главы.txt
    file_path = os.path.join("output", clean_book, str(agent_index), f"{clean_chap}.txt")
    
    # 3. Проверяем, существует ли файл
    if not os.path.exists(file_path):
        # Если файла нет, возвращаем None (чтобы скрипт оценки его пропустил)
        return None
    
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            
            # 4. Обработка заголовка (если мы записывали в файл метрики через --- METRICS ---)
            # Мы отрезаем служебную информацию, чтобы NER и COMET анализировали только текст
            if "--- METRICS ---" in content:
                # Разделяем по разделителю и берем последнюю часть
                parts = content.split("---------------")
                if len(parts) > 1:
                    content = parts[-1].strip()
            
            return content
            
    except Exception as e:
        print(f"Ошибка при чтении файла {file_path}: {e}")
        return None

In [40]:
import os
import re
import json


def calculate_meta_ner_accuracy(src_text, mt_text):
    """
    Считает точность перевода имен собственных (M-ETA NER Metric)
    """
    # 1. Извлекаем имена собственные из оригинала (PERSON, GPE, ORG)
    doc_en = nlp_en(src_text)
    entities_en = {ent.text for ent in doc_en.ents if ent.label_ in ["PERSON", "GPE", "ORG"]}
    
    if not entities_en:
        return 1.0 # Если имен нет, то и ошибок нет
    
    # 2. Извлекаем всё из перевода
    doc_ru = nlp_ru(mt_text)
    entities_ru = {ent.text for ent in doc_ru.ents}
    
    found_count = 0
    missing_entities = []

    for ent in entities_en:
        # Используем нечеткое сравнение, так как имена склоняются (Шерлок -> Шерлока)
        # Ищем, есть ли в русском тексте похожее слово
        has_match = any(fuzz.partial_ratio(ent, ru_ent) > 80 for ru_ent in entities_ru)
        
        if has_match:
            found_count += 1
        else:
            missing_entities.append(ent)
            
    # M-ETA Accuracy Score для имен собственных
    accuracy = found_count / len(entities_en)
    return accuracy, missing_entities
        

import comet
from comet.models import load_from_checkpoint, download_model

# Насильно заталкиваем функции в модуль comet, 
# так как библиотека evaluate ищет их именно там
comet.download_model = download_model
comet.load_from_checkpoint = load_from_checkpoint
import pandas as pd
# Теперь импортируем evaluate
import evaluate
comet_metric = evaluate.load('comet')

# Загружаем NER модели для M-ETA
nlp_en = spacy.load("en_core_web_sm")
nlp_ru = spacy.load("ru_core_news_sm")

def calculate_all_metrics(src_text, mt_text, ref_text):
    """
    Считает COMET-22 через Hugging Face и NER точность
    """
    # 1. Расчет COMET
    # Библиотека ожидает списки строк
    comet_results = comet_metric.compute(
        sources=[src_text], 
        predictions=[mt_text], 
        references=[ref_text]
    )
    comet_score = comet_results['mean_score']

    # 2. Расчет M-ETA (NER Точность)
    doc_en = nlp_en(ref_text)
    # Извлекаем имена собственные
    entities_en = {ent.text for ent in doc_en.ents if ent.label_ in ["PERSON", "GPE", "ORG"]}
    
    ner_acc = 1.0
    missing = []
    
    if entities_en:
        doc_ru = nlp_ru(mt_text)
        entities_ru = {ent.text for ent in doc_ru.ents}
        
        found = 0
        for name in entities_en:
            # Нечеткий поиск для учета склонений
            if any(fuzz.partial_ratio(name, ru_name) > 80 for ru_name in entities_ru):
                found += 1
            else:
                missing.append(name)
        ner_acc = found / len(entities_en)

    return {
        "comet_score": round(comet_score, 4),
        "ner_accuracy": round(ner_acc, 4),
        "missing_names": missing
    }

# --- Пример использования в твоем цикле оценки ---
def evaluate_agents(data_epub, agent_indices=[0]):
    all_data = []
    
    for agent_id in agent_indices:
        for book, chapters in data_epub.items():
            for chap_name, pairs in chapters.items():
                
                # Используем твою функцию чтения из файлов
                mt = get_agent_translation(book, agent_id, chap_name)
                if not mt: continue
                
                src = "\n".join([p['en'] for p in pairs])
                ref = "\n".join([p['ru'] for p in pairs])
                
                res = calculate_all_metrics(src, mt, ref)
                
                all_data.append({
                    "Agent": agent_id,
                    "Chapter": chap_name,
                    "COMET": res["comet_score"],
                    "NER_Acc": res["ner_accuracy"],
                    "Lost": ", ".join(res["missing_names"])
                })
                
    return pd.DataFrame(all_data)

Fetching 5 files: 100%|██████████| 5/5 [00:00<?, ?it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\andre\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
c:\Users\andre\AppData\Local\pypoetry\Cache\virtualenvs\ragtest-C3xehfLq-py3.11\Lib\site-packages\pytorch_lightning\core\saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [74]:
data3 = evaluate_agents(data,[1])

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs auto

In [71]:
data3

,Agent,Chapter,COMET,NER_Acc,Lost
0,1,CHAPTER I — A BEGGAR ON HORSEBACK,0.8319,0.1304,"по душе, Зовут, знакомые, эту юную леди, как в..."
1,1,CHAPTER II — THE HIGHLAND WRITER,0.8374,0.1626,"Пожалуй, Следовало бы поблагодарить, двести фу..."
2,1,CHAPTER III — I GO TO PILRIG,0.8787,0.1447,"могу только, по Ли-Уинд, среди дел политически..."
3,1,CHAPTER IV — LORD ADVOCATE PRESTONGRANGE,0.8590,0.1146,"этом уверен, сорок пятый год, ведь в этом, он,..."
4,1,CHAPTER V — IN THE ADVOCATE’S HOUSE,0.8782,0.1190,"как ребенка, шагал рядом, Видите, заметил в уг..."
5,1,CHAPTER VI — UMQUILE THE MASTER OF LOVAT,0.8864,0.1538,"Позор, Видите, чувствуя только, выполнить данн..."
6,1,CHAPTER VII — I MAKE A FAULT IN HONOUR,0.8704,0.1014,"где сбегали, что совершенно, — закричала она.,..."
7,1,CHAPTER VIII — THE BRAVO,0.8818,0.1359,"что виной тому мое воспитание, несколько тороп..."
8,1,CHAPTER IX — THE HEATHER ON FIRE,0.8749,0.2278,"что комендант окажется столь глуп или, Нийла, ..."
9,1,CHAPTER X — THE RED-HEADED MAN,0.8720,0.1327,"Тропинка, Нийла, как о возлюбленном, станете н..."


In [67]:
data2[:12]

,Agent,Chapter,COMET,NER_Acc,Lost
0,0,CHAPTER I — A BEGGAR ON HORSEBACK,0.8589,0.1401,"по душе, Зовут, знакомые, эту юную леди, как в..."
1,0,CHAPTER II — THE HIGHLAND WRITER,0.8408,0.1478,"Пожалуй, Следовало бы поблагодарить, двести фу..."
2,0,CHAPTER III — I GO TO PILRIG,0.8763,0.1447,"могу только, по Ли-Уинд, среди дел политически..."
3,0,CHAPTER IV — LORD ADVOCATE PRESTONGRANGE,0.8586,0.1146,"этом уверен, сорок пятый год, ведь в этом, он,..."
4,0,CHAPTER V — IN THE ADVOCATE’S HOUSE,0.8879,0.1190,"как ребенка, шагал рядом, Видите, заметил в уг..."
5,0,CHAPTER VI — UMQUILE THE MASTER OF LOVAT,0.8875,0.1657,"Позор, чувствуя только, выполнить данное, Фрэз..."
6,0,CHAPTER VII — I MAKE A FAULT IN HONOUR,0.8328,0.1106,"где сбегали, что совершенно, — закричала она.,..."
7,0,CHAPTER VIII — THE BRAVO,0.8845,0.1304,"что виной тому мое воспитание, несколько тороп..."
8,0,CHAPTER IX — THE HEATHER ON FIRE,0.8788,0.2222,"что комендант окажется столь глуп или, Нийла, ..."
9,0,CHAPTER X — THE RED-HEADED MAN,0.8733,0.1378,"Тропинка, Нийла, как о возлюбленном, станете н..."


In [70]:
data2['COMET'].mean()

np.float64(0.8701021505376344)

In [73]:
data3['COMET'].mean()

np.float64(0.8683641509433964)